In [2]:
!pip install -q transformers

In [40]:
import os
import json
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import ViltProcessor, ViltConfig, ViltForImagesAndTextClassification
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

train_json = "/kaggle/input/jsonfiles/multi_train.json"
val_json = "/kaggle/input/jsonfiles/multi_dev.json"
test_json = "/kaggle/input/jsonfiles/multi_test.json"
img_dir = "/kaggle/input/images/3k_image"
model_save_path = "/kaggle/working/ViLT_GOLD"

# label
label2id = {"Non-sarcasm": 0, "Sarcasm": 1}
id2label = {0: "Non-sarcasm", 1: "Sarcasm"}
num_labels = len(label2id)

# Load the base config first
base_config = ViltConfig.from_pretrained("dandelin/vilt-b32-mlm")

# Update the config for your specific classification task
config = ViltConfig(
    hidden_size=base_config.hidden_size,
    image_size=base_config.image_size,
    vocab_size=base_config.vocab_size,
    num_hidden_layers=base_config.num_hidden_layers,
    num_attention_heads=base_config.num_attention_heads,
    intermediate_size=base_config.intermediate_size,
    hidden_act=base_config.hidden_act,
    hidden_dropout_prob=base_config.hidden_dropout_prob,
    attention_probs_dropout_prob=base_config.attention_probs_dropout_prob,
    initializer_range=base_config.initializer_range,
    layer_norm_eps=base_config.layer_norm_eps,
    num_images=1, # Explicitly set num_images to 1
    num_labels=num_labels, # Set your number of labels
    id2label=id2label,
    label2id=label2id
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SarcasmDataset(Dataset):
    def __init__(self, json_file, img_dir, processor):
        with open(json_file, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        self.img_dir = img_dir
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['caption']
        image_path = os.path.join(self.img_dir, item['image'])
        image = Image.open(image_path).convert('RGB')
        label = label2id[item['label']]

        encoding = self.processor(image, text, padding="max_length", truncation=True, return_tensors="pt")
        encoding = {k: v.squeeze() for k, v in encoding.items()}
        encoding["labels"] = torch.tensor(label) # This label is already an integer ID
        return encoding

processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-mlm")

train_dataset = SarcasmDataset(train_json, img_dir, processor)
val_dataset = SarcasmDataset(val_json, img_dir, processor)

def collate_fn(batch):
    input_ids = torch.stack([x['input_ids'] for x in batch])
    attention_mask = torch.stack([x['attention_mask'] for x in batch])
    token_type_ids = torch.stack([x['token_type_ids'] for x in batch])
    pixel_values = [x['pixel_values'] for x in batch]
    labels_int = torch.stack([x['labels'] for x in batch]) # KEEP AS INTEGER LABELS

    # Removed one-hot encoding here.
    # labels_one_hot = torch.zeros(len(batch), num_labels, dtype=torch.float)
    # labels_one_hot.scatter_(1, labels_int.unsqueeze(1), 1)

    encoding = processor.image_processor.pad(pixel_values, return_tensors="pt")

    padded_pixel_values = encoding['pixel_values'].unsqueeze(1)
    padded_pixel_mask = encoding['pixel_mask'].unsqueeze(1)

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'token_type_ids': token_type_ids,
        'pixel_values': padded_pixel_values,
        'pixel_mask': padded_pixel_mask,
        'labels': labels_int # Pass the integer labels directly
    }

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

model = ViltForImagesAndTextClassification.from_pretrained(
    "dandelin/vilt-b32-mlm",
    config=config
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

model.train()
for epoch in range(5):
    print(f"\nEpoch {epoch+1}")

    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc="Training"):
        batch = {k: v.to(device) for k, v in batch.items()}
        # Do NOT cast labels to float for CrossEntropyLoss (it expects Long type)
        # batch["labels"] = batch["labels"].float() # REMOVED THIS LINE
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Train Loss: {avg_train_loss:.4f}")

    model.eval()
    val_loss = 0
    preds = []
    targets = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            batch = {k: v.to(device) for k, v in batch.items()}
            # Do NOT cast labels to float for CrossEntropyLoss
            # batch["labels"] = batch["labels"].float() # REMOVED THIS LINE
            outputs = model(**batch)
            val_loss += outputs.loss.item()

            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1).cpu().numpy()
            # Labels are already integer IDs from the dataset/collate_fn now
            labels = batch["labels"].cpu().numpy() # Get integer labels directly

            preds.extend(predictions)
            targets.extend(labels)

    avg_val_loss = val_loss / len(val_loader)
    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average='macro')
    precision = precision_score(targets, preds, average='macro')
    recall = recall_score(targets, preds, average='macro')

    print(f"Val Loss: {avg_val_loss:.4f} | Accuracy: {acc:.4f} | F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")

# Save model
model.save_pretrained(model_save_path)
processor.save_pretrained(model_save_path)

Some weights of ViltForImagesAndTextClassification were not initialized from the model checkpoint at dandelin/vilt-b32-mlm and are newly initialized: ['classifier.0.bias', 'classifier.0.weight', 'classifier.1.bias', 'classifier.1.weight', 'classifier.3.bias', 'classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Epoch 1


Training: 100%|██████████| 75/75 [00:25<00:00,  2.93it/s]


Train Loss: 0.6269


Validation: 100%|██████████| 25/25 [00:08<00:00,  3.09it/s]


...

Training: 100%|██████████| 75/75 [00:22<00:00,  3.30it/s]


Train Loss: 0.1288


Validation: 100%|██████████| 25/25 [00:07<00:00,  3.51it/s]


Val Loss: 0.9374 | Accuracy: 0.6400 | F1: 0.5960 | Precision: 0.5954 | Recall: 0.5968


[]

In [42]:
from sklearn.metrics import classification_report

test_dataset = SarcasmDataset(test_json, img_dir, processor)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader):
        labels = batch["labels"]
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds = torch.argmax(outputs.logits, dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print(classification_report(all_labels, all_preds, target_names=list(label2id.keys())))

100%|██████████| 25/25 [00:08<00:00,  3.03it/s]

              precision    recall  f1-score   support

 Non-sarcasm       0.76      0.75      0.75       135
     Sarcasm       0.49      0.51      0.50        65

    accuracy                           0.67       200
   macro avg       0.63      0.63      0.63       200
weighted avg       0.67      0.67      0.67       200

